# Recomendador de Filmes — Letterboxd

Sistema de recomendação baseado em conteúdo utilizando dados do Letterboxd.

### Como funciona
- Cada filme é representado por um vetor TF-IDF construído a partir de gênero, tema e diretor
- A similaridade entre filmes é calculada via **Cosseno**
- O score final combina **70% similaridade + 30% qualidade** (rating normalizado)
- Os top 20 similares de cada filme são salvos num **SQLite** de ~25MB

### Dataset
- Fonte: [Letterboxd Dataset — Kaggle](https://www.kaggle.com/datasets/gsimonx37/letterboxd)
- Arquivos utilizados: `movies.csv`, `crew.csv`, `genres.csv`, `themes.csv`, `posters.csv`

## 1. Imports e Configurações

In [3]:
import pandas as pd
import numpy as np
import sqlite3
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Configurações do modelo
RATING_MINIMO     = 2.5   # filmes abaixo disso são descartados
TOP_N_SIMILARES   = 20    # similares salvos por filme
PESO_SIMILARIDADE = 0.70  # peso da similaridade TF-IDF no score final
PESO_QUALIDADE    = 0.30  # peso do rating no score final
MAX_FEATURES_TFIDF = 5000 # tamanho do vocabulário TF-IDF
GENEROS_EXCLUIDOS = ["Documentary", "TV Movie"]  # categorias removidas

print("Imports carregados com sucesso!")

Imports carregados com sucesso!


## 2. Carregamento dos Dados

In [5]:
movies  = pd.read_csv("archive/movies.csv")
crew    = pd.read_csv("archive/crew.csv")
genres  = pd.read_csv("archive/genres.csv")
themes  = pd.read_csv("archive/themes.csv")
posters = pd.read_csv("archive/posters.csv")

print("Arquivos carregados:")
print(f"  movies.csv  → {len(movies):,} filmes")
print(f"  crew.csv    → {len(crew):,} membros")
print(f"  genres.csv  → {len(genres):,} registros")
print(f"  themes.csv  → {len(themes):,} registros")
print(f"  posters.csv → {len(posters):,} pôsteres")

Arquivos carregados:
  movies.csv  → 941,597 filmes
  crew.csv    → 4,720,183 membros
  genres.csv  → 1,046,849 registros
  themes.csv  → 125,641 registros
  posters.csv → 941,597 pôsteres


## 3. Limpeza e Filtros

Aplicamos três filtros:
1. **Rating mínimo de 2.5** — remove filmes sem avaliações ou muito mal avaliados
2. **Remove Documentários e TV Movies** — evita que contaminem as recomendações
3. **Remove filmes sem rating** — 850k dos 941k filmes não têm avaliações suficientes

In [7]:
# Filtro 1: rating mínimo
movies_ok = movies[movies["rating"] >= RATING_MINIMO].copy()
print(f"Após filtro de rating >= {RATING_MINIMO}: {len(movies_ok):,} filmes")

# Filtro 2: remover documentários e TV Movies
ids_remover = genres[
    genres["genre"].isin(GENEROS_EXCLUIDOS)
]["id"].unique()

movies_ok = movies_ok[~movies_ok["id"].isin(ids_remover)].copy()
print(f"Após remover {GENEROS_EXCLUIDOS}: {len(movies_ok):,} filmes")
print(f"\nFilmes prontos para o modelo: {len(movies_ok):,}")

Após filtro de rating >= 2.5: 86,497 filmes
Após remover ['Documentary', 'TV Movie']: 72,550 filmes

Filmes prontos para o modelo: 72,550


## 4. Construção dos Metadados

Juntamos gênero, tema e diretor em cada filme para montar o texto do TF-IDF.

In [9]:
# Diretores (apenas role = Director)
diretores = (
    crew[crew["role"] == "Director"][["id", "name"]]
    .rename(columns={"name": "diretor"})
    .groupby("id")["diretor"]
    .apply(lambda x: " ".join(x))
    .reset_index()
)

# Gêneros agrupados por filme (excluindo os removidos)
genres_agrupado = (
    genres[~genres["genre"].isin(GENEROS_EXCLUIDOS)]
    .groupby("id")["genre"]
    .apply(lambda x: " ".join(x))
    .reset_index()
)
genres_agrupado.columns = ["id", "generos"]

# Temas agrupados por filme
themes_agrupado = (
    themes
    .groupby("id")["theme"]
    .apply(lambda x: " ".join(x))
    .reset_index()
)
themes_agrupado.columns = ["id", "temas"]

# Montar DataFrame final
df = movies_ok[["id", "name", "date", "description", "rating"]].copy()
df = df.merge(diretores,        on="id", how="left")
df = df.merge(genres_agrupado,  on="id", how="left")
df = df.merge(themes_agrupado,  on="id", how="left")
df = df.merge(posters[["id", "link"]], on="id", how="left")

# Preencher nulos
for col in ["diretor", "generos", "temas", "description"]:
    df[col] = df[col].fillna("")

print(f"DataFrame final: {df.shape}")
df[["name", "generos", "temas", "diretor"]].head(3)

DataFrame final: (72550, 9)


,name,generos,temas,diretor
0,Barbie,Comedy Adventure,Humanity and the world around us Crude humor a...,Greta Gerwig
1,Parasite,Comedy Thriller Drama,Humanity and the world around us Intense viole...,Bong Joon-ho
2,Everything Everywhere All at Once,Science Fiction Adventure Comedy Action,Humanity and the world around us Moving relati...,Daniel Scheinert Daniel Kwan


## 5. Construção do Texto TF-IDF

Cada filme vira uma string com pesos diferentes por campo:
- **Gênero → 4x** (mais importante)
- **Tema → 2x** (médio)
- **Diretor → 1x** (complementar)

Descrição foi removida pois causava matches por texto genérico.

In [11]:
def montar_texto(row):
    partes = []
    if row["generos"]:
        partes.append((row["generos"] + " ") * 4)
    if row["temas"]:
        partes.append((row["temas"] + " ") * 2)
    if row["diretor"]:
        partes.append(row["diretor"])
    return " ".join(partes).strip()

df["texto"] = df.apply(montar_texto, axis=1)

# Exemplo
print("Exemplo — Parasite:")
print(df[df["name"] == "Parasite"]["texto"].values[0][:300])

Exemplo — Parasite:
Comedy Thriller Drama Comedy Thriller Drama Comedy Thriller Drama Comedy Thriller Drama  Humanity and the world around us Intense violence and sexual transgression Twisted dark psychological thriller Heartbreaking and moving family drama Enduring stories of family and marital drama Touching and sent


## 6. Cálculo de Similaridade

Transformamos o texto em vetores TF-IDF e calculamos a similaridade de cosseno entre todos os filmes em batches para não estourar a RAM.

> ⚠️ Esse processo leva alguns minutos (~10-15 min para 72k filmes).

In [13]:
print("Calculando TF-IDF...")
tfidf = TfidfVectorizer(max_features=MAX_FEATURES_TFIDF, stop_words="english")
matriz_tfidf = tfidf.fit_transform(df["texto"])
print(f"Matriz TF-IDF: {matriz_tfidf.shape}")

print("\nCalculando similaridades em batches...")
BATCH_SIZE = 1000
resultados = []
total = len(df)

for i in range(0, total, BATCH_SIZE):
    if i % 5000 == 0:
        print(f"  {i}/{total}...")
    batch = matriz_tfidf[i:i+BATCH_SIZE]
    sim = cosine_similarity(batch, matriz_tfidf)
    for j, scores in enumerate(sim):
        idx_filme = i + j
        top_indices = np.argsort(scores)[::-1]
        top_indices = [idx for idx in top_indices if idx != idx_filme][:TOP_N_SIMILARES]
        for rank, idx_similar in enumerate(top_indices):
            resultados.append({
                "filme_id":  df.iloc[idx_filme]["id"],
                "similar_id": df.iloc[idx_similar]["id"],
                "score": round(float(scores[idx_similar]), 2),
                "rank": rank + 1
            })

similares_df = pd.DataFrame(resultados)
print(f"\nTotal de pares calculados: {len(similares_df):,}")

Calculando TF-IDF...
Matriz TF-IDF: (72550, 5000)

Calculando similaridades em batches...
  0/72550...
  5000/72550...
  10000/72550...
  15000/72550...
  20000/72550...
  25000/72550...
  30000/72550...
  35000/72550...
  40000/72550...
  45000/72550...
  50000/72550...
  55000/72550...
  60000/72550...
  65000/72550...
  70000/72550...

Total de pares calculados: 1,451,000


## 7. Score de Qualidade

Combinamos a similaridade com o rating normalizado do filme.
Isso garante que filmes de baixa qualidade não apareçam nas recomendações mesmo sendo similares em conteúdo.

```
score_final = 70% × similaridade + 30% × rating_normalizado
```

In [15]:
# Normalizar rating entre 0 e 1
df["rating_norm"] = (
    (df["rating"] - df["rating"].min()) /
    (df["rating"].max() - df["rating"].min())
)
rating_dict = df.set_index("id")["rating_norm"].to_dict()

# Calcular score final
similares_df["rating_similar"] = similares_df["similar_id"].map(rating_dict)
similares_df["score_final"] = (
    PESO_SIMILARIDADE * similares_df["score"] +
    PESO_QUALIDADE    * similares_df["rating_similar"]
)

# Agrupar top N por filme ordenados pelo score final
similares_agrupado = (
    similares_df
    .sort_values(["filme_id", "score_final"], ascending=[True, False])
    .groupby("filme_id")
    .apply(lambda x: ",".join(x["similar_id"].astype(str)), include_groups=False)
    .reset_index()
)
similares_agrupado.columns = ["filme_id", "similares_ids"]

print(f"Similares agrupados: {len(similares_agrupado):,} filmes")

Similares agrupados: 72,550 filmes


## 8. Exportação para SQLite

Salvamos dois dados no banco:
- **filmes** — informações básicas de cada filme
- **similares** — top 20 similares por filme em formato comprimido

In [17]:
DB_PATH = "recomendador_final.db"

conn = sqlite3.connect(DB_PATH)

# Tabela de filmes
df[["id", "name", "date", "rating", "rating_norm", "link", "generos"]].to_sql(
    "filmes", conn, if_exists="replace", index=False
)

# Tabela de similares
similares_agrupado.to_sql(
    "similares", conn, if_exists="replace", index=False
)

# Índice para consultas rápidas
conn.execute("CREATE INDEX IF NOT EXISTS idx_filme_id ON similares(filme_id)")
conn.execute("VACUUM")
conn.close()

tamanho = os.path.getsize(DB_PATH) / (1024 * 1024)
print(f"Banco salvo em '{DB_PATH}'")
print(f"Tamanho: {tamanho:.1f} MB")

Banco salvo em 'recomendador_final.db'
Tamanho: 25.7 MB


## 9. Testes Finais

Validação das recomendações para alguns filmes conhecidos.

In [19]:
def testar(nome_filme, n=8):
    conn = sqlite3.connect(DB_PATH)
    filme = conn.execute("""
        SELECT f.id, f.name, s.similares_ids 
        FROM filmes f
        JOIN similares s ON f.id = s.filme_id
        WHERE f.name = ?
    """, (nome_filme,)).fetchone()

    if not filme:
        print(f"'{nome_filme}' não encontrado!")
        conn.close()
        return

    ids = filme[2].split(",")[:n]
    placeholders = ",".join(ids)
    similares = conn.execute(f"""
        SELECT name, rating, generos FROM filmes 
        WHERE id IN ({placeholders})
        ORDER BY rating DESC
    """).fetchall()
    conn.close()

    print(f"Recomendações para '{nome_filme}':")
    for i, (nome, rating, genero) in enumerate(similares, 1):
        print(f"  {i}. {nome} — ★ {rating} — {genero}")

testar("Hereditary")
print()
testar("Parasite")
print()
testar("Barbie")

Recomendações para 'Hereditary':
  1. The Tenant — ★ 3.86 — Thriller Mystery Horror
  2. In the Mouth of Madness — ★ 3.79 — Horror Thriller Mystery
  3. Barbarian — ★ 3.5 — Mystery Thriller Horror
  4. Dark Water — ★ 3.46 — Thriller Mystery Horror
  5. The Uninvited Guest — ★ 3.38 — Drama Horror Mystery Thriller
  6. Goodnight Mommy — ★ 3.37 — Mystery Thriller Horror
  7. Burnt Offerings — ★ 3.31 — Horror Thriller Mystery
  8. Lady in White — ★ 3.2 — Mystery Thriller Horror

Recomendações para 'Parasite':
  1. About Elly — ★ 4.11 — Drama
  2. Caché — ★ 4.06 — Drama Thriller Mystery
  3. Room — ★ 3.97 — Drama Thriller
  4. In the Bedroom — ★ 3.84 — Drama Thriller
  5. We Need to Talk About Kevin — ★ 3.82 — Drama Thriller
  6. Nocturnal Animals — ★ 3.73 — Drama Thriller
  7. The Killing of Two Lovers — ★ 3.65 — Thriller Drama
  8. Locke — ★ 3.6 — Thriller Drama

Recomendações para 'Barbie':
  1. Being John Malkovich — ★ 4.07 — Comedy Fantasy Drama
  2. Moonrise Kingdom — ★ 4.02 — Romance